# Cap Detection — Picture Feedback (camera only, no movement)

Turns on the **JETANK CSI camera**, grabs one frame **every 3 s**, sends it to the
**Roboflow** cloud model, and prints + draws any detected **bottle caps**.

**The robot does NOT move.** No `Robot()`, no motors, no arm — camera only.

Camera init is taken verbatim from `navigation.ipynb`; the Roboflow loop is ported
from `use_model_mac.py` (CSI camera instead of the Mac webcam, notebook widget
instead of `cv2.imshow`).

**Run order:** 1 Imports -> 2 Config -> 3 Camera init -> 4 Roboflow functions ->
5 Start picture feedback.  To stop: **Kernel -> Interrupt** (or the stop button),
then run **6 Release camera**.

## 1. Imports  (camera only — no Robot)

In [ ]:
import cv2
import numpy as np
import time
import base64
import json
from urllib import request as _rq, error as _er
import ipywidgets as widgets
from IPython.display import display
from jetbot import Camera, bgr8_to_jpeg     # NOTE: no Robot import — robot never moves

print('Imports OK')

## 2. Configuration

In [ ]:
# Camera
CAMERA_WIDTH  = 300
CAMERA_HEIGHT = 300

# Roboflow cap model (same as use_model_mac.py)
API_KEY              = "Ub1KVwtGHHdLLKRzoxdG"
API_URL              = "https://serverless.roboflow.com/kais-workspace-stbmo/workflows/detect-count-and-visualize-3"
CONFIDENCE_THRESHOLD = 0.8
INTERVAL             = 3.0            # seconds between pictures
TARGET_CLASS         = "bottle cap"
REQUEST_TIMEOUT      = 15             # per-request timeout; flaky wifi won't hang the loop

print('Config set. Picture every %.0fs, target = %r' % (INTERVAL, TARGET_CLASS))

## 3. Camera Init  (robust — from navigation.ipynb)

Releases any stale camera, then tries JetBot/GStreamer -> nvargus recovery -> OpenCV -> V4L2. Never hangs.

In [ ]:
import threading, subprocess as _sp

# Clean up any stale camera first
try:
    camera.unobserve_all()
    camera.stop()
except Exception:
    pass

camera       = None
_camera_mode = None


class _OpenCVCamera:
    """Thread-based camera shim that mimics the JetBot Camera API."""
    def __init__(self, device=0, backend=None, width=300, height=300):
        if backend is None:
            self._cap = cv2.VideoCapture(device)
        else:
            self._cap = cv2.VideoCapture(device, backend)
        if not self._cap.isOpened():
            raise RuntimeError('VideoCapture failed to open: ' + str(device))
        self._lock    = threading.Lock()
        self._running = True
        self._cbs     = []
        self._width   = width
        self._height  = height
        self.value    = np.zeros((height, width, 3), dtype=np.uint8)
        self._thread  = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()

    def _loop(self):
        while self._running:
            ok, frame = self._cap.read()
            if ok:
                frame = cv2.resize(frame, (self._width, self._height))
                with self._lock:
                    self.value = frame
                ch = {'new': frame}
                for cb in list(self._cbs):
                    try:
                        cb(ch)
                    except Exception:
                        pass

    def observe(self, callback, names='value'):
        if callback not in self._cbs:
            self._cbs.append(callback)

    def unobserve_all(self):
        self._cbs.clear()

    def stop(self):
        self._running = False
        try:
            self._cap.release()
        except Exception:
            pass


def _recover_nvargus():
    try:
        _sp.run(['fuser', '-k', '/dev/video0'], capture_output=True, timeout=3)
        _sp.run(['pkill', '-9', 'nvargus-daemon'], capture_output=True, timeout=3)
        _sp.Popen(['nvargus-daemon'], stdout=_sp.DEVNULL, stderr=_sp.DEVNULL)
        time.sleep(2.5)
        print('  nvargus-daemon restarted.')
    except Exception as _e:
        print('  nvargus recovery error:', _e)


print('Camera: Attempt 1 — JetBot/GStreamer ...')
try:
    camera       = Camera.instance(width=CAMERA_WIDTH, height=CAMERA_HEIGHT)
    _camera_mode = 'jetbot'
    print('  Ready.  Mode: jetbot/GStreamer')
except Exception as _e1:
    print('  Failed:', _e1)
    print('Camera: Attempt 2 — nvargus recovery + retry ...')
    try:
        _recover_nvargus()
        camera       = Camera.instance(width=CAMERA_WIDTH, height=CAMERA_HEIGHT)
        _camera_mode = 'jetbot'
        print('  Ready after recovery.  Mode: jetbot/GStreamer')
    except Exception as _e2:
        print('  Failed:', _e2)
        print('Camera: Attempt 3 — OpenCV VideoCapture(0) ...')
        try:
            camera       = _OpenCVCamera(device=0, width=CAMERA_WIDTH, height=CAMERA_HEIGHT)
            _camera_mode = 'opencv'
            print('  Ready.  Mode: OpenCV')
        except Exception as _e3:
            print('  Failed:', _e3)
            print('Camera: Attempt 4 — V4L2 direct /dev/video0 ...')
            try:
                camera       = _OpenCVCamera(device='/dev/video0', backend=cv2.CAP_V4L2,
                                             width=CAMERA_WIDTH, height=CAMERA_HEIGHT)
                _camera_mode = 'v4l2'
                print('  Ready.  Mode: V4L2 direct')
            except Exception as _e4:
                print('  Failed:', _e4)
                camera       = None
                _camera_mode = None
                print()
                print('ALL camera attempts failed. In a host terminal run:')
                print('  fuser -k /dev/video0')
                print('  pkill -9 nvargus-daemon')
                print('then restart the container and reopen this notebook.')

if camera is not None:
    print('Camera ready.  Mode:', _camera_mode)

## 4. Roboflow Functions  (from use_model_mac.py)

In [ ]:
def infer_frame(frame):
    ok, enc = cv2.imencode(".jpg", frame, [int(cv2.IMWRITE_JPEG_QUALITY), 90])
    if not ok:
        raise RuntimeError("Failed to JPEG-encode frame")
    image_b64 = base64.b64encode(enc.tobytes()).decode("utf-8")
    payload = {"api_key": API_KEY,
               "inputs": {"image": {"type": "base64", "value": image_b64},
                          "confidence": CONFIDENCE_THRESHOLD}}
    req = _rq.Request(url=API_URL,
                      data=json.dumps(payload).encode("utf-8"),
                      headers={"Content-Type": "application/json",
                               "Accept": "application/json",
                               "User-Agent": "python-urllib/3"},
                      method="POST")
    with _rq.urlopen(req, timeout=REQUEST_TIMEOUT) as resp:
        return json.loads(resp.read().decode("utf-8"))


def extract_predictions(result):
    outputs = result.get("outputs", [])
    if not outputs:
        return []
    preds = outputs[0].get("predictions", {})
    if isinstance(preds, dict):
        preds = preds.get("predictions", [])
    return preds or []


def side_of(x, frame_w):
    if x < frame_w / 3:     return "LEFT"
    if x > 2 * frame_w / 3: return "RIGHT"
    return "CENTER"


def draw_predictions(frame, predictions):
    for pred in predictions:
        x, y = pred["x"], pred["y"]
        w, h = pred["width"], pred["height"]
        x1, y1 = int(x - w / 2), int(y - h / 2)
        x2, y2 = int(x + w / 2), int(y + h / 2)
        if pred["class"] == TARGET_CLASS:
            color, thick = (0, 255, 0), 2
        else:
            color, thick = (0, 180, 0), 1
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, thick)
        cv2.putText(frame, "%s %.2f" % (pred["class"], pred["confidence"]),
                    (x1, max(y1 - 6, 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)
    return frame


def run_inference(frame):
    """Send one frame to Roboflow. Returns (annotated_frame, predictions)."""
    try:
        result = infer_frame(frame)
    except _er.HTTPError as e:
        print("HTTP %s: %s" % (e.code, e.read().decode("utf-8", "replace")[:200]))
        return frame, []
    except Exception as e:
        print("Inference failed:", e)
        return frame, []

    predictions = extract_predictions(result)
    if not predictions:
        print("No caps.")
        return frame, []
    fw = frame.shape[1]
    for i, p in enumerate(predictions):
        tag = ">>> CAP" if p["class"] == TARGET_CLASS else "    "
        print("%s [%d] %-12s conf=%.2f  %-6s  x=%.0f y=%.0f" %
              (tag, i, p["class"], p["confidence"], side_of(p["x"], fw), p["x"], p["y"]))
    return draw_predictions(frame, predictions), predictions


print("Roboflow functions defined.")

## 5. Start Picture Feedback

Grabs a frame every 3 s and detects caps. **To stop: Kernel -> Interrupt** (or the ◼ stop button).

In [ ]:
if camera is None:
    print("ERROR: camera not initialized — run the Camera Init cell first.")
else:
    cap_widget = widgets.Image(format='jpeg', width=CAMERA_WIDTH, height=CAMERA_HEIGHT)
    cap_label  = widgets.Label(value='starting...')
    display(widgets.VBox([cap_widget, cap_label]))
    print("Picture feedback ON — one frame every %.0fs. Interrupt the kernel to stop.\n" % INTERVAL)

    shot = 0
    try:
        while True:
            t0 = time.time()
            frame = camera.value
            if frame is None or getattr(frame, 'size', 0) == 0:
                time.sleep(0.2)
                continue
            frame = frame.copy()

            shot += 1
            print("--- picture %d @ %s ---" % (shot, time.strftime('%H:%M:%S')))
            annotated, preds = run_inference(frame)
            caps = [p for p in preds if p["class"] == TARGET_CLASS]

            cap_widget.value = bgr8_to_jpeg(annotated)
            cap_label.value  = ("caps: %d" % len(caps)) if caps else "no caps"
            if caps:
                print(">>> %d bottle cap(s): %s" %
                      (len(caps), ", ".join("%s @ %s" % (c["confidence"], side_of(c["x"], frame.shape[1])) for c in caps)))
            print()

            dt = time.time() - t0
            time.sleep(max(0.0, INTERVAL - dt))     # keep ~3s cadence
    except KeyboardInterrupt:
        print("Stopped by user (kernel interrupt).")

## 6. Release Camera

Run when done so the next session can open the camera.

In [ ]:
try:
    camera.unobserve_all()
    camera.stop()
    print("Camera released.")
except Exception as e:
    print("Release error:", e)